# 02. 成本控制与可靠性

Agent 的「无限循环」不是比喻——是真的会烧钱。成本控制和可靠性是生产 Agent 的基本功。

> **检查点**：能说出 Agent 的 4 个主要成本来源（LLM tokens、工具调用、检索、人工审批）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 成本管理器
from dataclasses import dataclass

@dataclass
class Budget:
    max_tokens: int; max_tool_calls: int; max_cost: float
    tokens_used: int = 0; tool_calls: int = 0; cost: float = 0.0
    
    def consume_tokens(self, n: int) -> bool:
        if self.tokens_used + n > self.max_tokens:
            return False
        self.tokens_used += n
        self.cost += n * 0.00001  # $0.01/1K tokens
        return True
    
    def consume_tool_call(self) -> bool:
        if self.tool_calls >= self.max_tool_calls:
            return False
        self.tool_calls += 1
        self.cost += 0.001
        return True
    
    def status(self) -> str:
        return (f"Tokens: {self.tokens_used}/{self.max_tokens}, "
                f"Tools: {self.tool_calls}/{self.max_tool_calls}, "
                f"Cost: ${self.cost:.4f}/{self.max_cost}")

# 重试与熔断
class RetryPolicy:
    def __init__(self, max_retries: int = 3, base_delay: float = 1.0):
        self.max_retries = max_retries; self.base_delay = base_delay
        self.failures = 0; self.circuit_open = False
    
    def should_retry(self, error: Exception) -> bool:
        self.failures += 1
        if self.failures >= self.max_retries:
            self.circuit_open = True
            return False
        return isinstance(error, (TimeoutError, ConnectionError))
    
    def delay(self) -> float:
        return self.base_delay * (2 ** (self.failures - 1))
    
    def reset(self):
        self.failures = 0; self.circuit_open = False

# 模拟 Agent 执行
budget = Budget(max_tokens=1000, max_tool_calls=5, max_cost=0.05)
retry = RetryPolicy(max_retries=3)

print("Agent execution:")
for step in range(1, 6):
    if retry.circuit_open:
        print(f"  Step {step}: CIRCUIT BREAKER OPEN — aborting")
        break
    if not budget.consume_tokens(150):
        print(f"  Step {step}: TOKEN BUDGET EXCEEDED")
        break
    if not budget.consume_tool_call():
        print(f"  Step {step}: TOOL CALL LIMIT REACHED")
        break
    print(f"  Step {step}: completed. {budget.status()}")

print(f"\nFinal: {budget.status()}")

Agent execution:
  Step 1: completed. Tokens: 150/1000, Tools: 1/5, Cost: $0.0025/0.05
  Step 2: completed. Tokens: 300/1000, Tools: 2/5, Cost: $0.0050/0.05
  Step 3: completed. Tokens: 450/1000, Tools: 3/5, Cost: $0.0075/0.05
  Step 4: completed. Tokens: 600/1000, Tools: 4/5, Cost: $0.0100/0.05
  Step 5: completed. Tokens: 750/1000, Tools: 5/5, Cost: $0.0125/0.05

Final: Tokens: 750/1000, Tools: 5/5, Cost: $0.0125/0.05


## 可靠性模式速查

| 模式 | 何时用 | 实现 |
|------|--------|------|
| Retry + Backoff | 瞬时故障（网络超时） | exponential backoff + jitter |
| Circuit Breaker | 连续失败 | 失败 N 次后熔断，定时半开探测 |
| Idempotency | 写操作（避免重复执行） | idempotency key + 结果缓存 |
| Timeout | 所有外部调用 | 每层独立超时，避免雪崩 |
| Rate Limit | 保护下游 | token bucket / sliding window |

> **检查点**：能区分 transient error（可重试）和 permanent error（不可重试），并举例。

## 真实 API 实验

预算管理实测（`cost_manager_demo.py --use-api`）：真实 token 计量下三种超限策略的行为差异。


In [3]:
SCRIPT = ROOT / 'agents' / 'production' / 'scripts' / 'cost_manager_demo.py'
import subprocess, sys
for strategy in ['hard-limit', 'warn']:
    print(f"===== strategy={strategy} =====")
    result = subprocess.run([sys.executable, str(SCRIPT), '--budget-tokens', '60', '--strategy', strategy,
                             '--steps', '5', '--use-api', '--verbose'], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)


===== strategy=hard-limit =====


step 1: tool_call budget → ok
step 1: +24 tokens (total 24/60) → ok
step 2: tool_call budget → ok
step 2: +24 tokens (total 48/60) → ok
step 3: tool_call budget → ok
🛑 token budget exceeded (72/60) — agent halted at step 3

Final: 72 tokens, 3 tool calls, strategy=hard-limit

===== strategy=warn =====


step 1: tool_call budget → ok
step 1: +24 tokens (total 24/60) → ok
step 2: tool_call budget → ok
step 2: +24 tokens (total 48/60) → ok
step 3: tool_call budget → ok
step 3: +24 tokens (total 72/60) → ⚠️ over token budget: 72/60
step 4: tool_call budget → ok
step 4: +24 tokens (total 96/60) → ⚠️ over token budget: 96/60
step 5: tool_call budget → ok
step 5: +24 tokens (total 120/60) → ⚠️ over token budget: 120/60

Final: 120 tokens, 5 tool calls, strategy=warn



## 练习

1. 实现 idempotency：给工具调用增加 idempotency_key，如果同一 key 已执行过，直接返回缓存结果。
2. 给 RetryPolicy 增加 jitter（随机抖动）：防止 thundering herd 问题。
3. 设计 Agent 的成本报告：按 task、按 tool、按 model 分维度展示费用。